In [1]:
import os
from dotenv import load_dotenv

#langchain

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


C:\Users\vikas\AppData\Local\Temp\ipykernel_12652\1933817996.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VARIABLE LOADED")

ENV VARIABLE LOADED


LOAD OUR DATA

In [4]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

DATA INGESTION

In [5]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")

documents = loader.load()

print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

#### LANGCHAIN DOCUMENT

Langchain processes everything in the form of documents

DOCUMENTS :

PAGE CONTENT -- the actual data

METADATA - extra information about the data

In [6]:
len(documents)

1

In [7]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [8]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [9]:
print(f"Total no. of characters in document: {len(documents[0].page_content)}")

Total no. of characters in document: 2597


SPLITTING OUR DATA

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [11]:
len(chunks)

9

NOW EACH SPLITTED CHUNK IS A DOCUMENT

In [12]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [13]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


In [14]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [15]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


EMBEDD OUR DATA

In [16]:
embeddingd_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL IS READY, THE NAME IS", embeddingd_model.model_name)

EMB MODEL IS READY, THE NAME IS jina-embeddings-v2-base-en


STORE DATA IN VECTOR DB

In [17]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddingd_model)

print("CHUNKS ARE STORED", vector_store.index.ntotal)

CHUNKS ARE STORED 9


WE NEVER STORED IT

In [28]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query : {test_query}")
for i,match in enumerate(top_matches, start=1):
 print(f"---Match {i}---")
 print(match.page_content)
 print()
# for doc, score in top_matches:
#     print(score)
#     print(doc.page_content)

Query : How many sick leaves employees get
---Match 1---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---Match 2---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### TOOL

In [19]:
retreiver = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)->str:
    """ 
    search the HR policy document for leave, work form home, probation,
    notice period, reimbursement, code of conduct, holiday or exit process
    """
    matching_chunk = retreiver.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunk)

### DATA RETRIVAL

LLM

In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature= 0)

llm.model_name

'openai/gpt-oss-120b'

In [21]:
test_response = llm.invoke("Hey is learing rag hard? reply in joke manner in one line")

test_response.content

'Learning RAG isn’t hard—it’s just trying to teach a robot to be a librarian who also moonlights as a stand‑up comic! 🤖📚😂'

### AI AGENT

LLM
TOOL
MEMORY

In [22]:
from langchain.agents import create_agent

In [38]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant working for Acmecrop.
    Always use the search_hr_policy tool to look up facta before answering .
    If the answer isn't in the search results, say you don't know "instead of guessing".
    """
)



In [24]:
def ask_hr_assistant(question:str)->str:
    """send a question to the RAG agent and print a nicely formatted answer."""
    print("="*80)
    print("QUESTION:" , question)

    response = hr_assistant.invoke({"messages":[{"role":"user", "content":question}]})
    answer = response['messages'][-1].content

    print("ANSWER : ", answer)
    print("="*80)
    print()
    return answer

In [25]:
print(ask_hr_assistant("tell me aboout leave policies"))

QUESTION: tell me aboout leave policies
ANSWER :  **Leave Policies – at a glance**

| Type of leave | Entitlement | How to use it | Key details |
|---------------|------------|---------------|--------------|
| **Annual (paid) leave** | 20 days per calendar year (full‑time) | Submit your request through the HR portal **at least 5 working days** before the intended start date. | • Up to **5 unused days** can be carried forward to the next year. <br>• Any remaining balance beyond the carry‑forward limit is forfeited at year‑end. |
| **Sick leave** | 10 paid days per year | Notify your manager and log the sick day in the HR system. | • A **medical certificate** is required if the sick leave exceeds **2 consecutive days**. |
| **Public holidays** | 12 days per year (company‑observed) | The holiday calendar is published by HR at the start of each year. | • If you are required to work on a public holiday, you receive **compensatory leave**. |
| **Leave carry‑forward** | Up to 5 days of unused

In [39]:
response = hr_assistant.invoke(
    {
        "messages":[
            {"role":"user",
             "content": "Tell me which org you work for"}
        ]
    }
)

In [40]:
response

{'messages': [HumanMessage(content='Tell me which org you work for', additional_kwargs={}, response_metadata={}, id='2d189488-51c0-4127-b256-68d58f37b401'),
  AIMessage(content='I’m the HR assistant here at **Acmecrop**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "Tell me which org you work for". The system says I\'m a friendly HR assistant working for Acmecrop. So answer: I work for Acmecrop. No need to search policy. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 202, 'total_tokens': 281, 'completion_time': 0.165573859, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.00807708, 'prompt_tokens_details': None, 'queue_time': 0.281057733, 'total_time': 0.173650939}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd52d-8769-77c0-a80a-8fb9fb

In [41]:
 response['messages'][-1].content

'I’m the HR assistant here at **Acmecrop**. How can I help you today?'

In [32]:
 response['messages'][-2].content

'1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n8. EXIT POLICY\nUpon resignation or termination, employees must complete a clearance process involving\nIT, Finance, and HR departments before their last working day.\nFull and final settlement, including any pending reimbursements and leave encashment,\nis processed within 45 days of the last working day.\n\n7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'

In [33]:
 response['messages'][-3].content

''

In [34]:
 response['messages'][-4].content

'Tell me how to apply for leave'